# ACP T4 Pipeline — Kaggle Dual T4 edition (2 × 15 GB)

**Before running**
1. Settings → Accelerator → **GPU T4 x2**. Settings → **Internet ON**.
2. Paste your HF token in cell 2 (`HF_TOKEN = ...`). The token must have accepted licenses for
   `meta-llama/Meta-Llama-3-8B` and `mistralai/Mistral-7B-Instruct-v0.3`.
3. Run cell 1 (pip) once. If you see import errors afterwards: *Run → Restart & clear outputs*, then run from cell 2.
4. For the full run, use **Save Version → Save & Run All (Commit)** so it survives the browser closing (12 h limit).

**Memory plan**
- `cuda:0` — student Llama-3-8B (4-bit NF4 + LoRA, ~6 GB) + MiniLM embedder. The frozen reference model for
  T1 entropy is the *same* student with LoRA adapters disabled (identical to the frozen base checkpoint), so no second 8B copy.
- `cuda:1` — teachers, **loaded one at a time** in 4-bit, used for generation, then fully unloaded.

In [ ]:
# ===================== CELL 1: INSTALL =====================
!pip install -q -U "transformers>=4.46,<5" "peft>=0.13" "accelerate>=1.0" "bitsandbytes>=0.45" "sentence-transformers>=3.0" "datasets>=3.0"

In [ ]:
# ===================== CELL 2: ENV, IMPORTS, LOGIN =====================
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

HF_TOKEN = "PASTE_YOUR_HF_TOKEN_HERE"   # <-- hardcode your token here

import gc, json, time, random, re, csv, math
from collections import Counter, defaultdict
from dataclasses import dataclass, field, replace, asdict
from typing import List, Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          get_linear_schedule_with_warmup)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from sklearn.cluster import MiniBatchKMeans
from sentence_transformers import SentenceTransformer
from huggingface_hub import login

if not HF_TOKEN.startswith('hf_') or 'PASTE' in HF_TOKEN:
    raise RuntimeError('Put your real Hugging Face token in HF_TOKEN above.')
os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

random.seed(42); np.random.seed(42); torch.manual_seed(42)

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Set Accelerator = GPU T4 x2 in notebook settings.')
N_GPUS = torch.cuda.device_count()
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB')

STUDENT_DEVICE = 0
TEACHER_DEVICE = 1 if N_GPUS > 1 else 0
if N_GPUS < 2:
    print('WARNING: only 1 GPU visible — teachers will share cuda:0 with the student (tight on a 15 GB T4).')

# T4 = compute capability 7.5: no native bf16 and no FlashAttention-2 -> fp16 + SDPA
COMPUTE_DTYPE = torch.float16
ATTN_IMPL = 'sdpa'

def bnb_4bit():
    return BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )

def free_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

def gpu_report(tag=''):
    parts = []
    for i in range(N_GPUS):
        parts.append(f'cuda:{i} alloc={torch.cuda.memory_allocated(i)/1e9:.1f}G '
                     f'reserved={torch.cuda.memory_reserved(i)/1e9:.1f}G')
    print(f'  [mem{(" " + tag) if tag else ""}] ' + ' | '.join(parts))

In [ ]:
# ===================== CELL 3: CONFIG =====================
SAVE_DIR = '/kaggle/working/anticollapse/t4_results/llama_en_real'
os.makedirs(SAVE_DIR, exist_ok=True)

STUDENT_ID = 'Qwen/Qwen2.5-7B'
TEACHER_SPECS = [                       # loaded/unloaded one at a time
    ('teacher_primary', 'Qwen/Qwen2.5-7B-Instruct'),
    ('teacher_mistral', 'mistralai/Mistral-7B-Instruct-v0.3'),
]
CRITIC_TOKENIZER_ID = 'Qwen/Qwen2.5-7B-Instruct'   # tokenizer only (tiny), kept for the lexical critic

# Run size. On 2xT4 in 4-bit, QUOTA=40 (400 candidates/round) is roughly 35-50 min/round.
# QUOTA=200 (2000 candidates/round) will NOT fit Kaggle's 12 h limit. 8 rounds matches your baseline table.
ROUNDS = 8
QUOTA = 40

@dataclass
class T4Config:
    USE_T4: bool = True
    LANGUAGE: str = 'en'
    SAVE_DIR: str = SAVE_DIR

    ROUTES: List[str] = field(default_factory=lambda: ['direct', 'paraphrase', 'expand', 'contrast', 'socratic'])

    # generation
    GENERATION_MAX_NEW_TOKENS_PLAIN: int = 256
    GENERATION_MAX_NEW_TOKENS_ROUTE: int = 320
    GENERATION_TEMPERATURE: float = 0.9
    GENERATION_TOP_P: float = 0.95
    GENERATION_BATCH_SIZE: int = 16       # auto-halves on OOM
    GENERATION_MAX_INPUT_TOKENS: int = 512
    USE_CHAT_TEMPLATE: bool = True        # instruction routes go through the teacher's chat template
    TEACHERS_PER_ROUTE: int = 2

    # correctness gate
    CORRECTNESS_MIN_ROUTES: int = 2
    CORRECTNESS_SIM_THRESHOLD: float = 0.55

    # diversity / route balance
    MIN_ROUTE_ENTROPY_BITS: float = 1.6
    MAX_ROUTE_SHARE: float = 0.45

    # Red-Queen critics
    CRITIC_NAMES: List[str] = field(default_factory=lambda: ['template', 'style', 'entropy_shape', 'lexical_diversity'])
    CRITIC_REJECT_VOTES_NEEDED: int = 2
    CRITIC_ROTATION_EVERY: int = 3

    # tripwires
    TRIPWIRE_MIN_ACCEPT_RATE: float = 0.05
    TRIPWIRE_MAX_REPETITION: float = 0.40
    TRIPWIRE_MIN_ROUTE_ENTROPY: float = 1.0

    # counterfactual calibration probes (never accepted / never trained on)
    COUNTERFACTUAL_RATE: float = 0.15

    # entropy scoring
    ENTROPY_BATCH_SIZE: int = 8
    ENTROPY_MAX_TOKENS: int = 150

    # training (same effective batch 16 as before: 2 x 8 instead of 4 x 4, to fit a T4)
    TRAIN_EPOCHS_PER_ROUND: int = 1
    TRAIN_BATCH_SIZE: int = 2
    TRAIN_GRAD_ACCUM: int = 8
    TRAIN_LR: float = 1e-4
    TRAIN_MAX_LEN: int = 512
    TRAIN_MAX_STEPS_PER_ROUND: int = 50
    ROUTE_UPWEIGHT_FACTOR: float = 2.0

t4_config = T4Config()
print(t4_config)

In [ ]:
# ===================== CELL 4: CANDIDATES + TEACHER GENERATION (load -> generate -> unload) =====================
@dataclass
class RouteCandidate:
    prompt: str
    output: str
    route: str
    teacher_name: str
    round_id: int
    prompt_id: int
    correctness_group: Optional[int] = None
    is_counterfactual: bool = False
    critic_votes: List[str] = field(default_factory=list)
    accepted: bool = False
    route_rank: Optional[int] = None
    entropy: Optional[float] = None
    cluster_id: Optional[int] = None
    embedding: Optional[np.ndarray] = field(default=None, repr=False)


ROUTE_PROMPT_TEMPLATES = {
    'direct': "{prompt}",
    'paraphrase': "Rewrite the following passage in different words while preserving its meaning:\n\n{prompt}",
    'expand': "Continue and elaborate on the following passage with additional relevant detail:\n\n{prompt}",
    'contrast': ("Write a passage on the same topic as the following, but from a contrasting "
                 "perspective or emphasis:\n\n{prompt}"),
    'socratic': "Pose and answer a probing question inspired by the following passage:\n\n{prompt}",
}


def load_teacher(model_id, device_idx):
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = 'left'
    tok.truncation_side = 'left'   # never cut off the end of the prompt / generation header
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb_4bit(), device_map={'': device_idx},
        torch_dtype=COMPUTE_DTYPE, attn_implementation=ATTN_IMPL, low_cpu_mem_usage=True,
    )
    m.eval()
    m.config.use_cache = True
    return m, tok


def build_route_prompt(route, prompt, tok, config):
    """Returns (text, add_special_tokens). 'direct' = raw continuation (as in the original);
    instruction routes use the instruct teacher's chat template."""
    content = ROUTE_PROMPT_TEMPLATES[route].format(prompt=prompt[:800])
    if route != 'direct' and config.USE_CHAT_TEMPLATE and getattr(tok, 'chat_template', None):
        text = tok.apply_chat_template([{'role': 'user', 'content': content}],
                                       tokenize=False, add_generation_prompt=True)
        return text, False   # template already contains BOS where needed
    return content, True


@torch.no_grad()
def _generate_batch(model, tok, texts, add_special, max_new_tokens, config, device):
    enc = tok(texts, return_tensors='pt', padding=True, truncation=True,
              max_length=config.GENERATION_MAX_INPUT_TOKENS, add_special_tokens=add_special).to(device)
    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=True,
        temperature=config.GENERATION_TEMPERATURE, top_p=config.GENERATION_TOP_P,
        pad_token_id=tok.pad_token_id, use_cache=True,
    )
    gen = out[:, enc['input_ids'].shape[1]:]
    res = [tok.decode(g, skip_special_tokens=True).strip() for g in gen]
    del enc, out, gen
    return res


def _generate_safe(model, tok, texts, add_special, max_new_tokens, config, device):
    """Halves the batch on OOM instead of losing the whole batch."""
    try:
        return _generate_batch(model, tok, texts, add_special, max_new_tokens, config, device)
    except torch.cuda.OutOfMemoryError:
        free_memory()
        if len(texts) == 1:
            print('  [gen warning] OOM on a single sequence, skipping it')
            return ['']
        mid = len(texts) // 2
        return (_generate_safe(model, tok, texts[:mid], add_special, max_new_tokens, config, device) +
                _generate_safe(model, tok, texts[mid:], add_special, max_new_tokens, config, device))


def generate_route_candidates(prompts, config, round_id):
    candidates = []
    device = torch.device(f'cuda:{TEACHER_DEVICE}')
    bs = config.GENERATION_BATCH_SIZE
    for teacher_name, model_id in TEACHER_SPECS:
        t0 = time.time()
        print(f'  Loading {teacher_name} ({model_id}) on cuda:{TEACHER_DEVICE} ...', flush=True)
        t_model, t_tok = load_teacher(model_id, TEACHER_DEVICE)
        gpu_report('teacher loaded')
        try:
            for route in config.ROUTES:
                max_tok = (config.GENERATION_MAX_NEW_TOKENS_PLAIN if route == 'direct'
                           else config.GENERATION_MAX_NEW_TOKENS_ROUTE)
                built = [build_route_prompt(route, p, t_tok, config) for p in prompts]
                texts = [b[0] for b in built]
                add_special = built[0][1]
                n_before = len(candidates)
                for i in range(0, len(texts), bs):
                    try:
                        outs = _generate_safe(t_model, t_tok, texts[i:i + bs], add_special, max_tok, config, device)
                    except Exception as e:
                        print(f'  [gen warning] round={round_id} batch={i} route={route} teacher={teacher_name}: {e}')
                        continue
                    for j, out in enumerate(outs):
                        if out:
                            candidates.append(RouteCandidate(
                                prompt=prompts[i + j], output=out, route=route, teacher_name=teacher_name,
                                round_id=round_id, prompt_id=i + j))
                print(f'    {teacher_name}/{route}: {len(candidates) - n_before} outputs', flush=True)
        finally:
            del t_model, t_tok
            free_memory()
            gpu_report('teacher unloaded')
        print(f'  {teacher_name} done in {(time.time() - t0) / 60:.1f} min', flush=True)
    return candidates


def get_route_distribution(candidates):
    return dict(Counter(c.route for c in candidates))

In [ ]:
# ===================== CELL 5: CORRECTNESS, DIVERSITY, ROTATION, COUNTERFACTUALS, CRITICS =====================
# ---- correctness (self-consistency across routes) ----
def correctness_gate(route_candidates, config):
    """Uses precomputed, L2-normalised c.embedding."""
    by_prompt = defaultdict(list)
    for c in route_candidates:
        by_prompt[c.prompt_id].append(c)
    for _, group in by_prompt.items():
        embs = np.stack([c.embedding for c in group])
        sim = embs @ embs.T
        n = len(group)
        assigned = [-1] * n
        nxt = 0
        for i in range(n):
            if assigned[i] != -1:
                continue
            assigned[i] = nxt
            for j in range(i + 1, n):
                if assigned[j] == -1 and sim[i, j] >= config.CORRECTNESS_SIM_THRESHOLD:
                    assigned[j] = nxt
            nxt += 1
        for c, cid in zip(group, assigned):
            c.correctness_group = cid
    return route_candidates


def filter_correct_candidates(route_candidates, min_routes_per_prompt=2):
    by_prompt = defaultdict(list)
    for c in route_candidates:
        by_prompt[c.prompt_id].append(c)
    kept = []
    for _, group in by_prompt.items():
        routes_per_cluster = defaultdict(set)
        for c in group:
            routes_per_cluster[c.correctness_group].add(c.route)
        valid = {cid for cid, rs in routes_per_cluster.items() if len(rs) >= min_routes_per_prompt}
        kept.extend(c for c in group if c.correctness_group in valid)
    return kept


# ---- diversity ----
def route_entropy(route_dist):
    total = sum(route_dist.values())
    if total == 0:
        return 0.0
    ent = 0.0
    for cnt in route_dist.values():
        if cnt:
            p = cnt / total
            ent -= p * np.log2(p)
    return float(ent)


def stamp_diversity_on_candidates(candidates):
    counts = Counter(c.route for c in candidates)
    rank_map = {r: i + 1 for i, (r, _) in enumerate(sorted(counts.items(), key=lambda kv: kv[1]))}
    for c in candidates:
        c.route_rank = rank_map.get(c.route, len(rank_map))
    return candidates


def interleave_by_route(cands, seed):
    """FIX: candidates arrive grouped teacher->route->prompt. Processing them in that order made the
    per-round share check reject almost everything from the first route. Round-robin across routes."""
    rng = random.Random(seed)
    buckets = defaultdict(list)
    for c in cands:
        buckets[c.route].append(c)
    keys = sorted(buckets)
    for k in keys:
        rng.shuffle(buckets[k])
    out = []
    while any(buckets[k] for k in keys):
        for k in keys:
            if buckets[k]:
                out.append(buckets[k].pop())
    return out


# ---- critic rotation ----
class CriticRotationScheduler:
    def __init__(self, config):
        self.all_critics = list(config.CRITIC_NAMES)
        self.rotate_every = config.CRITIC_ROTATION_EVERY
        self._forced_next_round = None

    def get_active_critics(self, round_id):
        if self._forced_next_round is not None and self._forced_next_round[0] == round_id:
            return list(self._forced_next_round[1])
        n = len(self.all_critics)
        if n <= 2:
            return list(self.all_critics)
        block = (round_id - 1) // max(1, self.rotate_every)
        return random.Random(block).sample(self.all_critics, max(2, n - 1))

    def rotate_on_tripwire(self, round_id, force_include):
        others = [c for c in self.all_critics if c not in force_include]
        active = list(force_include) + others[:max(0, len(self.all_critics) - len(force_include))]
        self._forced_next_round = (round_id + 1, active)

    def get_schedule_summary(self):
        lines = ['Critic rotation schedule (deterministic per block):']
        for r in range(1, self.rotate_every * 3 + 1):
            lines.append(f'  round {r}: {self.get_active_critics(r)}')
        return '\n'.join(lines)


# ---- counterfactual probes ----
NEGATION_TRIGGERS = [' is ', ' was ', ' are ', ' were ', ' can ', ' does ']

def _shuffle_sentences(text, rng):
    s = re.split(r'(?<=[.!?])\s+', text.strip())
    if len(s) < 3:
        return text
    rng.shuffle(s)
    return ' '.join(s)

def _flip_negation(text, rng):
    for trig in NEGATION_TRIGGERS:
        idx = text.find(trig)
        if idx != -1:
            return text[:idx] + ' ' + trig.strip() + ' not ' + text[idx + len(trig):]
    return text

def make_counterfactuals(route_candidates, config, seed=42):
    """FIX: the original rewrote c.output IN PLACE, so shuffled / negated text could be accepted and
    trained on. Now we build separate copies used only to measure how well the gates catch them."""
    rng = random.Random(seed)
    cfs = []
    for c in route_candidates:
        if rng.random() < config.COUNTERFACTUAL_RATE:
            method = rng.choice(['shuffle', 'negate'])
            new = _shuffle_sentences(c.output, rng) if method == 'shuffle' else _flip_negation(c.output, rng)
            if new != c.output:
                cfs.append(replace(c, output=new, is_counterfactual=True, entropy=None,
                                   embedding=None, cluster_id=None, critic_votes=[]))
    return cfs


# ---- Red-Queen critics ----
TEMPLATE_PATTERNS = [
    r'^as an ai\b', r'^i cannot\b', r'^in conclusion,?\s*$', r'^in summary,?\s*$', r'^overall,?\s*$',
    r"^it'?s important to note", r'^certainly!?\s*$', r'^sure,?\s*here', r'^title:\s*',
]

def critic_template(c, **kw):
    text = c.output.strip().lower()
    if any(re.search(p, text) for p in TEMPLATE_PATTERNS):
        return True
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    return len(lines) >= 3 and len(set(lines)) <= max(1, len(lines) // 3)

def critic_style(c, **kw):
    sents = [s for s in re.split(r'(?<=[.!?])\s+', c.output.strip()) if s]
    if len(sents) < 3:
        return False
    lengths = [len(s.split()) for s in sents]
    cv = float(np.std(lengths)) / (float(np.mean(lengths)) + 1e-8)
    return cv < 0.08 and len(sents) >= 5

def critic_entropy_shape(c, entropy_fn=None, **kw):
    e = c.entropy
    if e is None and entropy_fn is not None:
        try:
            e = entropy_fn(c.output)
        except Exception:
            return False
    return e is not None and e < 0.15

def critic_lexical_diversity(c, tokenizer=None, **kw):
    toks = tokenizer.encode(c.output, add_special_tokens=False) if tokenizer is not None else \
        [w.lower() for w in c.output.split()]
    if len(toks) < 5:
        return False
    return len(set(toks)) / len(toks) < 0.35

CRITIC_REGISTRY = {
    'template': critic_template, 'style': critic_style,
    'entropy_shape': critic_entropy_shape, 'lexical_diversity': critic_lexical_diversity,
}

def evaluate_all_critics(c, active_critics, tokenizer=None, entropy_fn=None):
    votes = []
    for name in active_critics:
        fn = CRITIC_REGISTRY.get(name)
        if fn is None:
            continue
        try:
            if fn(c, tokenizer=tokenizer, entropy_fn=entropy_fn):
                votes.append(name)
        except Exception:
            pass
    return votes

In [ ]:
# ===================== CELL 6: T4 ACCEPTANCE FILTER =====================
def t4_filter(route_candidates, round_id, config, rotation_scheduler, gate_fn, commit_fn,
              entropy_fn, prepare_fn, critic_tokenizer=None, existing_accepted=None):
    """
    a. counterfactual probes (separate copies; only measured, never accepted)
    b. T1/T2/T3 checks (side-effect free)
    c. per-round route diversity (on route-interleaved order)
    d. rotating Red-Queen critics
    e. global route balance + commit (dedup memory / T3 counts updated ONLY for accepted items)
    """
    stats = Counter()
    stats['input_candidates'] = len(route_candidates)
    if not route_candidates:
        return [], dict(stats)
    active_critics = rotation_scheduler.get_active_critics(round_id)
    stats['active_critics'] = ','.join(active_critics)

    # a. counterfactual probes
    cfs = make_counterfactuals(route_candidates, config, seed=round_id)
    if cfs:
        prepare_fn(cfs)   # entropy + embedding + cluster for the probes
        caught = 0
        for c in cfs:
            ok1, ok2, ok3 = gate_fn(c, round_id)
            votes = evaluate_all_critics(c, active_critics, tokenizer=critic_tokenizer, entropy_fn=entropy_fn)
            if not (ok1 and ok2 and ok3) or len(votes) >= config.CRITIC_REJECT_VOTES_NEEDED:
                caught += 1
        stats['cf_probes'] = len(cfs)
        stats['cf_caught'] = caught
        stats['cf_catch_rate'] = caught / len(cfs)

    # b. T1/T2/T3
    stage_b = []
    for c in route_candidates:
        ok1, ok2, ok3 = gate_fn(c, round_id)
        if not ok1:
            stats['reject_t1'] += 1
        elif not ok2:
            stats['reject_t2'] += 1
        elif not ok3:
            stats['reject_t3'] += 1
        else:
            stage_b.append(c)
    stats['pass_t1t2t3'] = len(stage_b)
    if not stage_b:
        return [], dict(stats)

    # c. per-round route diversity
    stage_b = interleave_by_route(stamp_diversity_on_candidates(stage_b), seed=round_id)
    stage_c, running = [], Counter()
    for c in stage_b:
        n_run = sum(running.values())
        share_if_added = (running[c.route] + 1) / (n_run + 1)
        if n_run >= 20 and share_if_added > config.MAX_ROUTE_SHARE:
            stats['reject_route_diversity'] += 1
            continue
        running[c.route] += 1
        stage_c.append(c)
    stats['pass_route_diversity'] = len(stage_c)
    if not stage_c:
        return [], dict(stats)

    # d. critics
    stage_d = []
    for c in stage_c:
        c.critic_votes = evaluate_all_critics(c, active_critics, tokenizer=critic_tokenizer, entropy_fn=entropy_fn)
        if len(c.critic_votes) >= config.CRITIC_REJECT_VOTES_NEEDED:
            stats['reject_critics'] += 1
            continue
        stage_d.append(c)
    stats['pass_critics'] = len(stage_d)
    if not stage_d:
        return [], dict(stats)

    # e. global route balance + commit
    existing = Counter(c.route for c in (existing_accepted or []))
    final = []
    for c in stage_d:
        total_existing = sum(existing.values()) + 1
        projected = (existing[c.route] + 1) / total_existing
        if total_existing >= 50 and projected > config.MAX_ROUTE_SHARE + 0.15:
            stats['reject_global_route_balance'] += 1
            continue
        ok, why = commit_fn(c)
        if not ok:
            stats[f'reject_commit_{why}'] += 1
            continue
        existing[c.route] += 1
        c.accepted = True
        final.append(c)

    stats['final_accepted'] = len(final)
    stats['final_route_entropy'] = route_entropy(get_route_distribution(final))
    stats['accept_rate'] = len(final) / max(1, stats['input_candidates'])
    return final, dict(stats)

In [ ]:
# ===================== CELL 7: DIAGNOSTICS + DASHBOARD =====================
class T4Diagnostics:
    def __init__(self, config):
        self.config = config
        self.rounds = []

    def log_round(self, round_id, candidates, accepted, stats, extra=None):
        entry = {'round_id': round_id, 'num_candidates': len(candidates), 'num_accepted': len(accepted),
                 **{k: v for k, v in stats.items() if not isinstance(v, (list, dict))}, **(extra or {})}
        self.rounds.append(entry)
        return entry

    def get_adaptation_actions(self):
        if not self.rounds:
            return []
        last, cfg, actions = self.rounds[-1], self.config, set()
        if last.get('accept_rate', 1.0) < cfg.TRIPWIRE_MIN_ACCEPT_RATE:
            actions.add('rotate_critics')
        if last.get('final_route_entropy', 99.0) < cfg.TRIPWIRE_MIN_ROUTE_ENTROPY:
            actions.add('rotate_critics')
        if last.get('repetition_accepted', 0.0) > cfg.TRIPWIRE_MAX_REPETITION:   # was defined but never used
            actions.add('rotate_critics')
        return sorted(actions)

    def get_metrics_for_plotting(self):
        keys = ['num_candidates', 'num_accepted', 'accept_rate', 'final_route_entropy', 'reject_t1', 'reject_t2',
                'reject_t3', 'reject_critics', 'reject_route_diversity', 'reject_global_route_balance']
        d = {'round_id': [r['round_id'] for r in self.rounds]}
        for k in keys:
            d[k] = [r.get(k, 0) for r in self.rounds]
        return d

    def export(self, save_dir, verbose=True):
        os.makedirs(save_dir, exist_ok=True)
        jp = os.path.join(save_dir, 't4_diagnostics.json')
        with open(jp, 'w') as f:
            json.dump(self.rounds, f, indent=2, default=float)
        if self.rounds:
            cp = os.path.join(save_dir, 't4_diagnostics.csv')
            keys = sorted({k for r in self.rounds for k in r})
            with open(cp, 'w', newline='') as f:
                w = csv.DictWriter(f, fieldnames=keys)
                w.writeheader()
                for r in self.rounds:
                    w.writerow(r)
            if verbose:
                print(f'  Diagnostics exported: {jp}, {cp}')


def plot_t4_dashboard(data, config, save_dir, show=True, dpi=300):
    os.makedirs(save_dir, exist_ok=True)
    rounds = data.get('round_id', [])
    if not rounds:
        print('  [dashboard] no data'); return
    fig, axes = plt.subplots(2, 3, figsize=(16, 9)); axes = axes.flatten()
    def _p(ax, key, title, color):
        ax.plot(rounds, data.get(key, []), marker='o', color=color)
        ax.set_title(title); ax.set_xlabel('Round'); ax.grid(True, alpha=0.3)
    _p(axes[0], 'num_candidates', 'Candidates (after correctness)', 'gray')
    _p(axes[1], 'num_accepted', 'Candidates Accepted', 'darkorange')
    _p(axes[2], 'accept_rate', 'Accept Rate', 'seagreen')
    _p(axes[3], 'final_route_entropy', 'Route Entropy (bits)', 'purple')
    ax = axes[4]
    for key, label in [('reject_t1', 'T1'), ('reject_t2', 'T2'), ('reject_t3', 'T3'), ('reject_critics', 'Critics'),
                       ('reject_route_diversity', 'Route Div.'), ('reject_global_route_balance', 'Global Bal.')]:
        ax.plot(rounds, data.get(key, []), marker='.', label=label)
    ax.set_title('Rejections by Gate'); ax.set_xlabel('Round'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    axes[5].axis('off')
    summary = ['T4 Dashboard Summary', f'Rounds logged: {len(rounds)}',
               f'Last accept rate: {data["accept_rate"][-1]:.3f}',
               f'Last route entropy: {data["final_route_entropy"][-1]:.3f} bits']
    axes[5].text(0.05, 0.5, '\n'.join(summary), fontsize=11, family='monospace', va='center',
                 bbox=dict(boxstyle='round', facecolor='lightyellow'))
    plt.suptitle('ACP T4 Diagnostics Dashboard', fontsize=14, fontweight='bold'); plt.tight_layout()
    out = os.path.join(save_dir, 't4_dashboard.png')
    plt.savefig(out, dpi=dpi, bbox_inches='tight')
    plt.show() if show else plt.close(fig)
    print(f'  Dashboard saved: {out}')

In [ ]:
# ===================== CELL 8: ROUTE-AWARE TRAINING (QLoRA, fp16 autocast + GradScaler) =====================
class RouteAwareDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts, self.tok, self.max_len = texts, tokenizer, max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        ids = self.tok(self.texts[i], add_special_tokens=True, truncation=True,
                       max_length=self.max_len - 1)['input_ids']
        return ids + [self.tok.eos_token_id]   # FIX: teach the student to stop


def make_collate(pad_id):
    def _collate(batch):
        L = max(len(x) for x in batch)   # FIX: dynamic padding instead of always padding to 512
        ids = torch.full((len(batch), L), pad_id, dtype=torch.long)
        mask = torch.zeros((len(batch), L), dtype=torch.long)
        for i, x in enumerate(batch):
            ids[i, :len(x)] = torch.tensor(x); mask[i, :len(x)] = 1
        labels = ids.clone()
        labels[mask == 0] = -100          # FIX: mask by attention (pad == eos for Llama-3 would mask real EOS)
        return {'input_ids': ids, 'attention_mask': mask, 'labels': labels}
    return _collate


def _route_weights(candidates, upweight_factor):
    counts = Counter(c.route for c in candidates)
    mx = max(counts.values())
    w = {r: 1.0 + (upweight_factor - 1.0) * (1.0 - n / mx) for r, n in counts.items()}
    return [w[c.route] for c in candidates]


def _make_scaler():
    try:
        return torch.amp.GradScaler('cuda')
    except Exception:
        return torch.cuda.amp.GradScaler()


def train_round_t4(model, tokenizer, candidates, config):
    if not candidates:
        print('  [train] no candidates, skipping'); return
    device = next(p for p in model.parameters() if p.requires_grad).device
    bs, accum = config.TRAIN_BATCH_SIZE, config.TRAIN_GRAD_ACCUM
    max_steps = config.TRAIN_MAX_STEPS_PER_ROUND
    num_samples = min(len(candidates) * config.TRAIN_EPOCHS_PER_ROUND, max_steps * bs * accum)
    sampler = WeightedRandomSampler(_route_weights(candidates, config.ROUTE_UPWEIGHT_FACTOR),
                                    num_samples=num_samples, replacement=True)
    ds = RouteAwareDataset([c.output for c in candidates], tokenizer, config.TRAIN_MAX_LEN)
    loader = DataLoader(ds, batch_size=bs, sampler=sampler, collate_fn=make_collate(tokenizer.pad_token_id))

    total_steps = max(1, math.ceil(len(loader) / accum))
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=config.TRAIN_LR)
    sched = get_linear_schedule_with_warmup(opt, max(1, total_steps // 10), total_steps)
    scaler = _make_scaler()

    model.train()
    model.config.use_cache = False
    step, micro, loss_sum = 0, 0, 0.0
    opt.zero_grad(set_to_none=True)

    def _opt_step():
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        opt.zero_grad(set_to_none=True)

    t0 = time.time()
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast('cuda', dtype=torch.float16):
            out = model(**batch)
        scaler.scale(out.loss / accum).backward()
        loss_sum += out.loss.item(); micro += 1
        del out, batch
        if micro % accum == 0:
            _opt_step(); step += 1
            if step >= total_steps:
                break
    if micro % accum != 0 and step < total_steps:   # FIX: don't drop the trailing partial accumulation
        _opt_step(); step += 1
    model.eval()
    free_memory()
    # FIX: avg over micro-batches actually run (old code divided by len(loader) even after early break)
    print(f'  [train] steps={step} avg_loss={loss_sum / max(1, micro):.4f} '
          f'micro_batches={micro} pool={len(candidates)} time={(time.time() - t0) / 60:.1f} min')

In [ ]:
# ===================== CELL 9: LOAD STUDENT (4-bit + LoRA), EMBEDDER, DATA =====================
tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID, quantization_config=bnb_4bit(), device_map={'': STUDENT_DEVICE},
    torch_dtype=COMPUTE_DTYPE, attn_implementation=ATTN_IMPL, low_cpu_mem_usage=True,
)
base.config.use_cache = False
base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
base.enable_input_require_grads()
model = get_peft_model(base, LoraConfig(
    r=16, lora_alpha=32, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
))
model.print_trainable_parameters()
model.eval()
print(f'Student loaded: {STUDENT_ID} (4-bit) on cuda:{STUDENT_DEVICE}')
print('Reference model = student with adapters disabled (frozen base weights, no extra copy).')

critic_tokenizer = AutoTokenizer.from_pretrained(CRITIC_TOKENIZER_ID)

embedder = SentenceTransformer('all-MiniLM-L6-v2', device=f'cuda:{STUDENT_DEVICE}')
print('Embedder loaded')
gpu_report('after student')

print('Streaming 2000 seed documents from FineWeb sample-10BT...', flush=True)
ds_stream = load_dataset('HuggingFaceFW/fineweb', name='sample-10BT', split='train', streaming=True)
raw_data = []
for x in ds_stream:
    t = x['text'].strip()
    if len(t) > 50:
        raw_data.append(t)
    if len(raw_data) >= 2000:
        break
print(f'Loaded {len(raw_data)} seed documents')

In [ ]:
# ===================== CELL 10: T1 / T2 / T3 GATES + METRICS =====================
# ---------- T1: entropy under the frozen reference (= base weights, adapters off) ----------
@torch.no_grad()
def calculate_entropy_batch(texts, batch_size=None):
    batch_size = batch_size or t4_config.ENTROPY_BATCH_SIZE
    model.eval()
    res = []
    old_side = tokenizer.padding_side
    tokenizer.padding_side = 'right'   # FIX: plain forward() doesn't derive positions from the mask
    dev = torch.device(f'cuda:{STUDENT_DEVICE}')
    try:
        with model.disable_adapter():
            for i in range(0, len(texts), batch_size):
                enc = tokenizer(texts[i:i + batch_size], return_tensors='pt', truncation=True,
                                max_length=t4_config.ENTROPY_MAX_TOKENS, padding=True).to(dev)
                labels = enc['input_ids'].masked_fill(enc['attention_mask'] == 0, -100)
                with torch.autocast('cuda', dtype=torch.float16):
                    logits = model(**enc, use_cache=False).logits
                sl = logits[:, :-1, :].float()
                lb = labels[:, 1:]
                loss = torch.nn.functional.cross_entropy(sl.reshape(-1, sl.size(-1)), lb.reshape(-1),
                                                         reduction='none', ignore_index=-100).view(lb.shape)
                m = (lb != -100).float()
                seq = (loss * m).sum(1) / m.sum(1).clamp(min=1)
                res.extend((seq / 0.6931).cpu().tolist())
                del enc, logits, sl, loss
    finally:
        tokenizer.padding_side = old_side
    return res


def calculate_entropy(text):
    return calculate_entropy_batch([text])[0]


# ---------- T2: tail + novelty ----------
SEED_TAIL_FRACTION = 0.10

def compute_tail_floor(texts, alpha=0.8):
    toks = []
    for s in texts:
        toks.extend(tokenizer.encode(s))
    counts = Counter(toks)
    sv = sorted(counts.items(), key=lambda x: x[1])
    k = max(1, int(len(sv) * SEED_TAIL_FRACTION))
    tail = set(t for t, _ in sv[:k])
    mass = sum(1 for t in toks if t in tail) / max(len(toks), 1)
    return mass * alpha, tail

SEED_TAIL_FLOOR, tail_set = compute_tail_floor(raw_data[:500])
print(f'Seed Tail Floor: {SEED_TAIL_FLOOR:.4f}')

seen_3grams = set()
for s in raw_data[:200]:
    tk = tokenizer.encode(s)
    for i in range(len(tk) - 2):
        seen_3grams.add(tuple(tk[i:i + 3]))

def t2_metrics(s):
    tok = tokenizer.encode(s)
    if not tok:
        return 0, 0, []
    tail = sum(1 for t in tok if t in tail_set) / len(tok) * 100
    ngrams = [tuple(tok[i:i + 3]) for i in range(len(tok) - 2)]
    nov = sum(1 for g in ngrams if g not in seen_3grams) / len(ngrams) * 100 if ngrams else 0
    return tail, nov, ngrams

def t2_gate(s):
    t, n, g = t2_metrics(s)
    tail_ok = (t >= SEED_TAIL_FLOOR * 10)       # unchanged from your original thresholds
    passed = tail_ok and ((n >= 6) or (t >= 0.5 and n >= 3))
    return passed, t, n, g


# ---------- T3: dedup + domain balance ----------
def embed_texts(texts):
    return embedder.encode(texts, batch_size=128, show_progress_bar=False,
                           normalize_embeddings=True, convert_to_numpy=True)

NUM_CLUSTERS = 8
cluster_model = MiniBatchKMeans(n_clusters=NUM_CLUSTERS, batch_size=64, random_state=42, n_init=10)
init_emb = embed_texts(raw_data[:500])
cluster_model.fit(init_emb)
counts_c = np.bincount(cluster_model.predict(init_emb), minlength=NUM_CLUSTERS)
target_dist = (counts_c + 1) / (counts_c.sum() + NUM_CLUSTERS)
cluster_counts = np.zeros(NUM_CLUSTERS)

DEDUP_THRESHOLD = 0.94
DEDUP_MEMORY = 2000
committed_embs = np.zeros((0, init_emb.shape[1]), dtype=np.float32)

def is_semantic_dup(emb):
    if committed_embs.shape[0] == 0:
        return False
    return bool((committed_embs @ emb).max() >= DEDUP_THRESHOLD)

def t3_check(cid):
    total = cluster_counts.sum()
    if total < 50:
        return True, 'warmup'
    cur = (cluster_counts + 1) / (total + 1e-8 + NUM_CLUSTERS)
    new = cur.copy(); new[cid] += 1 / (total + 1)
    kl = np.sum(new * np.log((new + 1e-8) / (target_dist + 1e-8)))
    return (kl <= 1.5), ('ok' if kl <= 1.5 else 'domain_skew')


def prepare_candidates(cands):
    """Batch-compute entropy, embedding and cluster once (was one forward/encode per candidate)."""
    if not cands:
        return cands
    ents = calculate_entropy_batch([c.output for c in cands])
    embs = embed_texts([c.output for c in cands])
    clus = cluster_model.predict(embs)
    for c, e, v, k in zip(cands, ents, embs, clus):
        c.entropy, c.embedding, c.cluster_id = float(e), v.astype(np.float32), int(k)
    return cands


current_LOW, current_HIGH = 0.0, 10.0

def t1t2t3_check(c, round_id):
    """Side-effect free (FIX: the original added to dedup memory / T3 counts even for candidates
    later rejected by critics or route balance)."""
    if c.entropy is None or not (current_LOW <= c.entropy <= current_HIGH):
        return False, False, False
    if not t2_gate(c.output)[0]:
        return True, False, False
    if is_semantic_dup(c.embedding):
        return True, False, False
    ok3, _ = t3_check(c.cluster_id)
    return True, True, ok3

def commit_accept(c):
    """Called only for finally-accepted candidates; re-checks against items accepted earlier this round."""
    global committed_embs
    if is_semantic_dup(c.embedding):
        return False, 'dup'
    ok3, _ = t3_check(c.cluster_id)
    if not ok3:
        return False, 't3'
    committed_embs = np.vstack([committed_embs, c.embedding[None]])
    cluster_counts[c.cluster_id] += 1
    return True, 'ok'

print('T1/T2/T3 gates ready')
print(f'T3 target distribution: {target_dist.round(3)}')


# ---------- metrics ----------
def heaps_metric(texts):
    toks = []
    for s in texts:
        toks.extend(tokenizer.encode(s))
    if len(toks) < 2:
        return 1.0, 0, 0
    N, V = len(toks), len(set(toks))
    return np.log(V) / np.log(N), V, N

def tail_mass_metric(texts):
    toks = []
    for s in texts:
        toks.extend(tokenizer.encode(s))
    return sum(1 for t in toks if t in tail_set) / len(toks) * 100 if toks else 0

def repetition_metric(texts):
    toks = []
    for s in texts:
        toks.extend(tokenizer.encode(s))
    if len(toks) < 2:
        return 0
    bg = [tuple(toks[i:i + 2]) for i in range(len(toks) - 1)]
    return sum(n - 1 for n in Counter(bg).values() if n > 1) / len(bg)

metrics_log = {k: {'heaps': [], 'tail_mass': [], 'repetition': [], 'entropy_mean': [], 'entropy_std': []}
               for k in ('generated', 'accepted')}

def log_metrics(cands, label, round_id, global_dataset):
    texts = [c.output for c in cands]
    beta, V, N = heaps_metric(texts if label == 'generated' else global_dataset + texts)
    tail, rep = tail_mass_metric(texts), repetition_metric(texts)
    ents = [c.entropy for c in cands if c.entropy is not None]
    if len(ents) > 100:   # FIX: random sample, not the first 100 (which were all one teacher/route)
        ents = random.Random(round_id).sample(ents, 100)
    em, es = (float(np.mean(ents)), float(np.std(ents))) if ents else (0.0, 0.0)
    print(f'\nMETRICS ({label}) ROUND {round_id + 1}')
    print(f'  Heaps beta: {beta:.3f} | V={V} | N={N}')
    print(f'  Tail Mass:  {tail:.3f}\n  Repetition: {rep:.3f}\n  Entropy:    {em:.3f} +/- {es:.3f}')
    for k, v in zip(['heaps', 'tail_mass', 'repetition', 'entropy_mean', 'entropy_std'], [beta, tail, rep, em, es]):
        metrics_log[label][k].append(float(v))
    return rep

print('Metrics ready')

In [ ]:
# ===================== CELL 11: MAIN LOOP =====================
rotation_scheduler = CriticRotationScheduler(t4_config)
t4_diagnostics = T4Diagnostics(t4_config)
print(rotation_scheduler.get_schedule_summary())

unused_indices = list(range(len(raw_data)))
np.random.shuffle(unused_indices)
all_t4_accepted = []

def draw_prompts(quota):
    global unused_indices
    if len(unused_indices) < quota:
        unused_indices = list(range(len(raw_data)))
        np.random.shuffle(unused_indices)
    return [raw_data[unused_indices.pop()] for _ in range(quota)]

def save_checkpoint(r, global_dataset):
    t4_diagnostics.export(SAVE_DIR, verbose=False)
    with open(os.path.join(SAVE_DIR, 'metrics_log.json'), 'w') as f:
        json.dump(metrics_log, f, indent=2)
    with open(os.path.join(SAVE_DIR, 'accepted.jsonl'), 'w') as f:
        for c in all_t4_accepted:
            d = asdict(c); d.pop('embedding', None)
            f.write(json.dumps(d, default=float) + '\n')
    model.save_pretrained(os.path.join(SAVE_DIR, 'lora_adapter'))
    print(f'  [checkpoint] round {r + 1} saved to {SAVE_DIR}')

def run_t4(rounds=ROUNDS, quota=QUOTA):
    global current_LOW, current_HIGH, committed_embs
    global_dataset = []
    run_t0 = time.time()
    for r in range(rounds):
        committed_embs = committed_embs[-DEDUP_MEMORY:]
        print(f'\n{"=" * 80}\nROUND {r + 1} / {rounds}\n{"=" * 80}')
        t0 = time.time()
        prompts = draw_prompts(quota)

        # 1. generation: teacher A load -> generate -> unload, then teacher B
        cands = generate_route_candidates(prompts, t4_config, round_id=r + 1)
        n_generated = len(cands)
        print(f'Generated: {n_generated} route candidates')
        if not cands:
            print('WARNING: no candidates generated this round'); continue

        # 2. entropy/embedding/cluster for every candidate (batched, student GPU)
        prepare_candidates(cands)
        log_metrics(cands, 'generated', r, global_dataset)

        # 3. dynamic entropy band
        valid = [c.entropy for c in cands if 0 < c.entropy < 10]
        if len(valid) > 10:
            current_LOW, current_HIGH = float(np.percentile(valid, 20)), float(np.percentile(valid, 80))
        else:
            current_LOW, current_HIGH = 0.0, 10.0
        print(f'Entropy band: {current_LOW:.3f} - {current_HIGH:.3f}')

        # 4. correctness gate
        cands = filter_correct_candidates(correctness_gate(cands, t4_config), t4_config.CORRECTNESS_MIN_ROUTES)
        print(f'After correctness gate: {len(cands)}')

        # 5. T4 filter
        accepted, stats = t4_filter(
            route_candidates=cands, round_id=r + 1, config=t4_config, rotation_scheduler=rotation_scheduler,
            gate_fn=t1t2t3_check, commit_fn=commit_accept, entropy_fn=calculate_entropy,
            prepare_fn=prepare_candidates, critic_tokenizer=critic_tokenizer, existing_accepted=all_t4_accepted,
        )
        print(f'  t4 stats: {stats}')
        all_t4_accepted.extend(accepted)
        print(f'\nT4 accepted this round: {len(accepted)}')
        dist = get_route_distribution(accepted)
        print(f'Route entropy: {route_entropy(dist):.3f} bits (max {np.log2(len(t4_config.ROUTES)):.3f})')
        print(f'Route distribution: {dist}')

        rep_acc = log_metrics(accepted, 'accepted', r, global_dataset)
        t4_diagnostics.log_round(r + 1, cands, accepted, stats,
                                 extra={'num_generated': n_generated, 'repetition_accepted': rep_acc})
        actions = t4_diagnostics.get_adaptation_actions()
        if actions:
            print(f'  Tripwire adaptation: {actions}')
            if 'rotate_critics' in actions:
                rotation_scheduler.rotate_on_tripwire(r + 1, ['template', 'style'])

        if accepted:
            global_dataset.extend(c.output for c in accepted)
        else:
            print('WARNING: Nothing accepted this round')

        # 6. route-aware training
        if all_t4_accepted:
            print(f'\nTraining on {len(all_t4_accepted)} cumulative samples...')
            train_round_t4(model, tokenizer, all_t4_accepted, t4_config)
        free_memory()
        save_checkpoint(r, global_dataset)

        dt = (time.time() - t0) / 60
        print(f'Round {r + 1} took {dt:.1f} min | total dataset: {len(global_dataset)}')
        if r == 0:
            est = dt * rounds
            print(f'\n[TIMING] Estimated full run: ~{est:.0f} min ({est / 60:.1f} h). '
                  f'Kaggle sessions stop at 12 h — lower ROUNDS/QUOTA if this is too close.\n')
    print(f'\nTotal runtime: {(time.time() - run_t0) / 60:.1f} min')
    return global_dataset

In [ ]:
# ===================== CELL 12: RUN =====================
final_dataset = run_t4()
print(f'\nExperiment complete! Final dataset: {len(final_dataset)} samples')

In [ ]:
# ===================== CELL 13: PLOTS, DASHBOARD, TABLE =====================
def plot_all_metrics(metrics_log):
    n = len(metrics_log['accepted']['heaps'])
    if n == 0:
        print('No rounds to plot'); return
    rounds = list(range(1, n + 1))
    titles = {'heaps': 'Heaps Beta (Vocabulary Growth)', 'tail_mass': 'Tail Mass (%)',
              'repetition': 'Bigram Repetition Rate', 'entropy_mean': 'Entropy Mean', 'entropy_std': 'Entropy Std'}
    fig, axes = plt.subplots(2, 3, figsize=(16, 10)); axes = axes.flatten()
    for idx, m in enumerate(titles):
        ax = axes[idx]
        ax.plot(rounds, metrics_log['generated'][m][:n], marker='o', label='Generated', color='steelblue')
        ax.plot(rounds, metrics_log['accepted'][m], marker='s', label='Accepted (T4)', color='darkorange')
        ax.set_xlabel('Round'); ax.set_ylabel(m); ax.set_title(titles[m]); ax.legend(); ax.grid(True, alpha=0.3)
    ax = axes[-1]; ax.axis('off')
    lines = ['T4 Experiment Summary', f'Rounds: {n}',
             f'Final Heaps beta (accepted): {metrics_log["accepted"]["heaps"][-1]:.3f}',
             f'Final Repetition (accepted): {metrics_log["accepted"]["repetition"][-1]:.3f}',
             f'Avg Entropy (accepted): {metrics_log["accepted"]["entropy_mean"][-1]:.3f}']
    ax.text(0.1, 0.5, '\n'.join(lines), transform=ax.transAxes, fontsize=12, va='center', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    plt.suptitle(f'ACP T4: Llama-3-8B on English FineWeb (dual teacher, {n} rounds, Kaggle 2xT4)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    out = os.path.join(SAVE_DIR, 't4_metrics_panel.png')
    plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()
    print(f'Metrics panel saved to {out}')

plot_all_metrics(metrics_log)

t4_diagnostics.export(SAVE_DIR)
plot_t4_dashboard(t4_diagnostics.get_metrics_for_plotting(), t4_config, SAVE_DIR, show=True, dpi=300)

print('\n' + '=' * 90)
print('RESULTS COMPARISON: T4 vs T1-T3 Baseline (Paper Table 1)')
print('=' * 90)
baseline = {
    'heaps': [0.771, 0.784, 0.784, 0.778, 0.776, 0.773, 0.768, 0.763],
    'repetition': [0.288, 0.297, 0.313, 0.269, 0.217, 0.268, 0.303, 0.331],
}
acc_per_round = [r.get('num_accepted', 0) for r in t4_diagnostics.rounds]
header = (f"{'Round':<6} | {'Heaps(T1-T3)':<13} | {'Heaps(T4)':<11} | {'Rep(T1-T3)':<11} | "
          f"{'Rep(T4)':<9} | {'T4 Accepted':<12}")
print(header); print('-' * len(header))
for i in range(len(metrics_log['accepted']['heaps'])):
    bh = f"{baseline['heaps'][i]:.3f}" if i < len(baseline['heaps']) else '—'
    br = f"{baseline['repetition'][i]:.3f}" if i < len(baseline['repetition']) else '—'
    acc = acc_per_round[i] if i < len(acc_per_round) else '—'
    print(f"R{i + 1:<5} | {bh:<13} | {metrics_log['accepted']['heaps'][i]:<11.3f} | {br:<11} | "
          f"{metrics_log['accepted']['repetition'][i]:<9.3f} | {acc}")
print('=' * 90)
print('Heaps beta >= baseline => vocabulary diversity preserved; Repetition <= baseline => less repetition.')
print(f'All outputs in: {SAVE_DIR}')